In [ ]:
# ALL TYPES typing
# typing-3.7.4.3

In [3]:
# TypeVar из модуля typing нужен, чтобы создавать обобщённые (generic) типы и связывать типы в
# сигнатуре функции: например, «какой тип передали на вход — такой же вернём на выход».
# Ниже 10 практических примеров с пояснениями.

from typing import TypeVar

In [ ]:
# 1. Функция-«идентичность»: вернуть тот же тип, что и на входе
# Зачем: тип результата строго зависит от типа аргумента. Статический анализатор не даст перепутать типы.

T = TypeVar("T")

def identity(x: T) -> T:
    return x

a: int = identity(10)          # T = int
b: str = identity("hello")     # T = str

a

10

In [11]:
# 2. Фабрика экземпляра по классу (с сохранением конкретного типа)
# Зачем: без TypeVar пришлось бы возвращать object или использовать Any,
# и терялась бы конкретная типизация.

from typing import Type, TypeVar

T = TypeVar("T")

def make_instance(cls: Type[T]) -> T:
    return cls()

class Dog:
    def bark(self) -> None:
        print("woof")

dog: Dog = make_instance(Dog)
dog.bark()  # mypy знает, что это Dog, а не просто object
print(type(dog))

woof
<class '__main__.Dog'>


In [12]:
dd = Dog()
dd.bark()
print(type(dd))

woof
<class '__main__.Dog'>


In [13]:
# 3. Функция преобразования с сохранением типа контейнера
# Зачем: мы говорим анализатору: «в списке был тип T, после преобразования в
# нём всё ещё тип T». Это полезно, когда функция не меняет тип элементов.
from typing import List, TypeVar

T = TypeVar("T")

def map_list(items: List[T], fn) -> List[T]:
    return [fn(x) for x in items]

nums: List[int] = [1, 2, 3]
result: List[int] = map_list(nums, lambda x: x * 2)

In [14]:
# 4. Ограниченный TypeVar: только подтипы определённого класса
# Зачем: bound=Animal разрешает передавать любые подтипы Animal,
# но запрещает всё остальное. Это частый случай в Airflow/Pydantic-подобных
# конструкциях, где нужно работать с семейством классов.
from typing import TypeVar, List

class Animal:
    pass

class Dog(Animal):
    pass

class Cat(Animal):
    pass

T = TypeVar("T", bound=Animal)

def feed(animals: List[T]) -> None:
    for a in animals:
        # a гарантированно имеет интерфейс Animal
        ...

feed([Dog(), Cat()])      # OK
feed(["not an animal"])   # mypy: ошибка

In [ ]:
# Type[T] используют, когда нужно типизировать сам класс (а не его экземпляр) — например, передать класс как 
# аргумент или вернуть его из функции. Ниже 10 практических примеров с короткими пояснениями.
from typing import Type